In [1]:
%load_ext cuml.accel
# %run /workspace/alvin/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
%run /mnt/d/Users/Admin/Projects/dso/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
import os
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, ConcatDataset
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from cuml.manifold import TSNE, UMAP
from joblib import Parallel, delayed
from tqdm import tqdm

In [2]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Uncomment only if you need 100% determinism and can handle errors
    # torch.use_deterministic_algorithms(True, warn_only=True)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    os.environ["PYTHONHASHSEED"] = str(seed)

def worker_init_fn(worker_id):
    """DataLoader worker init for reproducibility"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [3]:
# workspace = "/workspace/alvin/SAR_ML"
workspace = "/mnt/d/Users/Admin/Projects/dso/SAR_ML"
data_workspace = os.path.join(workspace, "data/SAMPLE")

In [4]:
with open(os.path.join(workspace, "weights/SSR/SAMPLE_synth_gmm_cache.pkl"), "rb") as f:
    gmm_cache = pickle.load(f)

In [5]:
no_aug_synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

SSR_synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), SSRAugmentation(gmm_cache, alpha=0.6, beta=0.4, apply_prob=0.5, gaussian_noise = True, mu_s = 0.0, sigma_s = 0.3), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

In [6]:
train_ds = SSR_synth_ds
test_ds  = meas_ds

ds_dict = {"train" : train_ds, "test": test_ds}
dataset_sizes = {"train" : len(train_ds), "test": len(test_ds)}

In [7]:
# seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]
seed_lst = [777, 849, 1000, 1111, 1234]

train_loss = []
# val_loss = []
train_acc =[]
# val_acc = []

for i, seed in enumerate(seed_lst):
    start = time.perf_counter()
    print(f"Training Run {i}: seed {seed}")

    set_seed(seed)

    dataloaders = {
        "train": DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=8, pin_memory = True, persistent_workers=True, prefetch_factor=4, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
        # "val": DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
        "test": DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory = True, persistent_workers=True, prefetch_factor=4, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed))
    }
    # load pre-trained model
    model = models.resnet18(weights = None)

    # Replace final layer for the number of classes
    model.fc = nn.Sequential(
        nn.Dropout(p = 0.4),
        nn.Linear(model.fc.in_features, len(train_ds.class_to_idx))
    )
    
    # Insert Dropout2d after each residual block
    dropout_rate = 0.1
    for layer in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for block in layer:
            block.add_module('block_dropout', nn.Dropout2d(p=dropout_rate))
            def make_forward(b, orig_fwd):
                def forward(x):
                    out = orig_fwd(x)
                    return b.block_dropout(out)
                return forward
            block.forward = make_forward(block, block.forward)

    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss() # most common used nn for classification problems
    optimizer = optim.AdamW(model.parameters(), lr = 3e-4, weight_decay = 2e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max = 200, eta_min = 3e-7)
        
    # move model to GPU
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    history = {
        "train_loss": [],
        # "val_loss" : [],
        "train_acc": [],
        # "val_acc" : []
    }
    
    # Training loops
    num_epochs = 200
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}")
        if epoch == 0:
            print(f"First layer mean: {model.conv1.weight.data.mean():.6f}")
        for phase in ["train"]:
            if phase == "train":
                model.train()
            else:
                model.eval()
    
            running_loss = 0.0
            running_corrects = 0 # correct predictions
    
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
    
                optimizer.zero_grad() # clear the gradient from previous iteration
    
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels) # check if output and labels match
    
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                        # scheduler.step() # scheduler here if OneCycleLR
    
                running_loss += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()
    
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]
            
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)
    
            print(f"{phase} Loss: {epoch_loss:.10f} Acc: {epoch_acc:.10f}")
    
        scheduler.step()
        print(f"Epoch {epoch} LR: {scheduler.get_last_lr()[0]:.10f}")
        
    print("Training complete!")
    
    train_loss.append(history["train_loss"])
    # val_loss.append(history["val_loss"])
    train_acc.append(history["train_acc"])
    # val_acc.append(history["val_acc"])
    
    torch.save(model.state_dict(), os.path.join(workspace, f"weights/SSR/Extras/mc_dropout/SSR_w_noise/rn18_seed{seed}_b16.pth"))

    end = time.perf_counter()
    elapsed = end - start
    print(f"Training time for Seed {seed}: {elapsed:.2f}s")

train_loss = np.array(train_loss)
# val_loss = np.array(val_loss)
train_acc = np.array(train_acc)
# val_acc = np.array(val_acc)

Training Run 0: seed 777
Epoch 0
First layer mean: 0.000157
train Loss: 2.2231379608 Acc: 0.1955390335
Epoch 0 LR: 0.0002999815
Epoch 1
train Loss: 1.6025740985 Acc: 0.3985130112
Epoch 1 LR: 0.0002999261
Epoch 2
train Loss: 0.9432330578 Acc: 0.6527881041
Epoch 2 LR: 0.0002998336
Epoch 3
train Loss: 0.5000349628 Acc: 0.8327137546
Epoch 3 LR: 0.0002997043
Epoch 4
train Loss: 0.3310872868 Acc: 0.8996282528
Epoch 4 LR: 0.0002995381
Epoch 5
train Loss: 0.2414173490 Acc: 0.9263940520
Epoch 5 LR: 0.0002993350
Epoch 6
train Loss: 0.1677545232 Acc: 0.9516728625
Epoch 6 LR: 0.0002990950
Epoch 7
train Loss: 0.1331762818 Acc: 0.9576208178
Epoch 7 LR: 0.0002988184
Epoch 8
train Loss: 0.1258334252 Acc: 0.9605947955
Epoch 8 LR: 0.0002985050
Epoch 9
train Loss: 0.1443651120 Acc: 0.9524163569
Epoch 9 LR: 0.0002981551
Epoch 10
train Loss: 0.0924006991 Acc: 0.9717472119
Epoch 10 LR: 0.0002977686
Epoch 11
train Loss: 0.1030657456 Acc: 0.9687732342
Epoch 11 LR: 0.0002973457
Epoch 12
train Loss: 0.071469078

# No Aug

In [8]:
meas_dataloader = DataLoader(
    meas_ds, batch_size=16, shuffle=False, 
    num_workers=8, pin_memory = True, persistent_workers=True, 
    prefetch_factor=4
                             )

for k in [10**2, 200, 400, 800, 10**3]:
    print(f"\nEvaluating with k={k} forward passes for MC Dropout...")

    seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    test_acc_lst = []

    for i, seed in enumerate(seed_lst):
        print(f"Evaluating Run {i}: seed {seed}")
        set_seed(seed)
        
        # Reconstruct architecture exactly as trained
        new_model = models.resnet18(weights=None)
        new_model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
        )
        
        dropout_rate = 0.1
        for layer in [new_model.layer1, new_model.layer2, new_model.layer3, new_model.layer4]:
            for block in layer:
                block.add_module('block_dropout', nn.Dropout2d(p=dropout_rate))
                def make_forward(b, orig_fwd):
                    def forward(x):
                        out = orig_fwd(x)
                        return b.block_dropout(out)
                    return forward
                block.forward = make_forward(block, block.forward)

        new_model.load_state_dict(torch.load(
            os.path.join(workspace, f"weights/SSR/Extras/mc_dropout/wo_aug/rn18_seed{seed}_b16.pth"),
            map_location=device
        ))
        
        new_model = new_model.to(device)
        new_model.eval()
        for m in new_model.modules():
            if isinstance(m, nn.Dropout2d):
                m.train()

        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in meas_dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # k forward passes, average probabilities, then classify
                preds_mc = torch.stack([new_model(inputs) for _ in range(k)], dim=0)
                mean_probs = torch.softmax(preds_mc, dim=-1).mean(0)
                _, preds = torch.max(mean_probs, 1)

                correct += torch.sum(preds == labels).item()
                total += labels.size(0)

        test_acc = correct / total
        print(f"Test Accuracy: {test_acc:.4f}")
        test_acc_lst.append(test_acc)


Evaluating with k=100 forward passes for MC Dropout...
Evaluating Run 0: seed 10
Test Accuracy: 0.9175
Evaluating Run 1: seed 42
Test Accuracy: 0.8691
Evaluating Run 2: seed 100
Test Accuracy: 0.8007
Evaluating Run 3: seed 123
Test Accuracy: 0.8498
Evaluating Run 4: seed 666
Test Accuracy: 0.8015
Evaluating Run 5: seed 777
Test Accuracy: 0.8743
Evaluating Run 6: seed 849
Test Accuracy: 0.8320
Evaluating Run 7: seed 1000
Test Accuracy: 0.8409
Evaluating Run 8: seed 1111
Test Accuracy: 0.8706
Evaluating Run 9: seed 1234
Test Accuracy: 0.8476

Evaluating with k=200 forward passes for MC Dropout...
Evaluating Run 0: seed 10
Test Accuracy: 0.9175
Evaluating Run 1: seed 42
Test Accuracy: 0.8729
Evaluating Run 2: seed 100
Test Accuracy: 0.8007
Evaluating Run 3: seed 123
Test Accuracy: 0.8483
Evaluating Run 4: seed 666
Test Accuracy: 0.7993
Evaluating Run 5: seed 777
Test Accuracy: 0.8781
Evaluating Run 6: seed 849
Test Accuracy: 0.8297
Evaluating Run 7: seed 1000
Test Accuracy: 0.8401
Evalua

In [11]:
# mean is 0.8512, seed 123 is the closest to the mean accuracy for wo aug

# SSR w Gaussian

In [9]:
meas_dataloader = DataLoader(
    meas_ds, batch_size=16, shuffle=False, 
    num_workers=8, pin_memory = True, persistent_workers=True, 
    prefetch_factor=4
    )

for k in [10**2, 200, 400, 800, 10**3]:
    print(f"\nEvaluating with k={k} forward passes for MC Dropout...")

    seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    test_acc_lst = []

    for i, seed in enumerate(seed_lst):
        print(f"Evaluating Run {i}: seed {seed}")
        set_seed(seed)
        
        # Reconstruct architecture exactly as trained
        new_model = models.resnet18(weights=None)
        new_model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
        )
        
        dropout_rate = 0.1
        for layer in [new_model.layer1, new_model.layer2, new_model.layer3, new_model.layer4]:
            for block in layer:
                block.add_module('block_dropout', nn.Dropout2d(p=dropout_rate))
                def make_forward(b, orig_fwd):
                    def forward(x):
                        out = orig_fwd(x)
                        return b.block_dropout(out)
                    return forward
                block.forward = make_forward(block, block.forward)

        new_model.load_state_dict(torch.load(
            os.path.join(workspace, f"weights/SSR/Extras/mc_dropout/SSR_w_noise/rn18_seed{seed}_b16.pth"),
            map_location=device
        ))
        
        new_model = new_model.to(device)
        new_model.eval()
        for m in new_model.modules():
            if isinstance(m, nn.Dropout2d):
                m.train()

        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in meas_dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # k forward passes, average probabilities, then classify
                preds_mc = torch.stack([new_model(inputs) for _ in range(k)], dim=0)
                mean_probs = torch.softmax(preds_mc, dim=-1).mean(0)
                _, preds = torch.max(mean_probs, 1)

                correct += torch.sum(preds == labels).item()
                total += labels.size(0)

        test_acc = correct / total
        print(f"Test Accuracy: {test_acc:.4f}")
        test_acc_lst.append(test_acc)


Evaluating with k=100 forward passes for MC Dropout...
Evaluating Run 0: seed 10
Test Accuracy: 0.9197
Evaluating Run 1: seed 42
Test Accuracy: 0.9279
Evaluating Run 2: seed 100
Test Accuracy: 0.9108
Evaluating Run 3: seed 123
Test Accuracy: 0.9138
Evaluating Run 4: seed 666
Test Accuracy: 0.8833
Evaluating Run 5: seed 777
Test Accuracy: 0.9011
Evaluating Run 6: seed 849
Test Accuracy: 0.9138
Evaluating Run 7: seed 1000
Test Accuracy: 0.8922
Evaluating Run 8: seed 1111
Test Accuracy: 0.9152
Evaluating Run 9: seed 1234
Test Accuracy: 0.9212

Evaluating with k=200 forward passes for MC Dropout...
Evaluating Run 0: seed 10
Test Accuracy: 0.9182
Evaluating Run 1: seed 42
Test Accuracy: 0.9294
Evaluating Run 2: seed 100
Test Accuracy: 0.9071
Evaluating Run 3: seed 123
Test Accuracy: 0.9138
Evaluating Run 4: seed 666
Test Accuracy: 0.8818
Evaluating Run 5: seed 777
Test Accuracy: 0.8996
Evaluating Run 6: seed 849
Test Accuracy: 0.9130
Evaluating Run 7: seed 1000
Test Accuracy: 0.8914
Evalua

In [10]:
# seed 100 is the closest to the mean accuracy for SSR with noise